# Create snow pillow comparison dataset

Mirrors `../compare_to_NorSWE/1_create_NorSWE_comparison_dataset.ipynb`, using the combined public snow station archive built in `0_download_and_preprocess_all_snow_pillow_data.ipynb`.

Steps:
1. Open the snow pillow Zarr, add water year / DOWY coordinates, restrict to the config's water years (full overlap with the SAR dataset)
2. Select stations: de-duplicate AWDB "MSNT" copies of native CCSS/BC stations, drop stations without SWE
3. QC the daily SWE series (spike removal, data-density and gap checks, seasonal-snowpack check)
4. Compute per-water-year max SWE value and %-of-max SWE timings → `data/comparison_datasets/<version>/max_snow_pillow_swe_timing.zarr`
5. Extract a 25×25 pixel chip (80 m pixels, ±1 km) from the global SAR runoff onset Zarr around each usable station → `data/comparison_datasets/<version>/runoff_onset_snow_pillow_station_chips.zarr`

In [ ]:
import xarray as xr
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import easysnowdata
import numpy as np
import rasterio
import os
import contextily as ctx
import shutil
from global_snowmelt_runoff_onset.config import Config

# The dataset version drives every output path below, so a v10 run cannot overwrite
# the v9 products (or vice versa). Switching versions = editing this one config path.
config = Config('config/global_config_v10.txt')
VERSION = config.version

DATA_DIR = Path("data/snow_pillows")
SNOW_PILLOW_ZARR_FILEPATH = DATA_DIR / "snow_pillows.zarr"

COMPARISON_DATA_DIR = Path('data/comparison_datasets') / VERSION
MAX_SWE_TIMING_OUTPUT_ZARR_FILEPATH = COMPARISON_DATA_DIR / 'max_snow_pillow_swe_timing.zarr'
RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH = COMPARISON_DATA_DIR / 'runoff_onset_snow_pillow_station_chips.zarr'

# --- temporal overlap of the SAR dataset and the ongoing snow pillow archives ---
# From the config so a version bump picks up its water years (v9: WY2015-2024,
# v10: WY2015-2025) instead of silently reusing the previous window.
TARGET_WATER_YEARS = [int(water_year) for water_year in config.water_years]

In [ ]:
snow_pillows_ds = xr.open_zarr(SNOW_PILLOW_ZARR_FILEPATH)
snow_pillows_ds

In [ ]:
snow_pillows_ds.coords['water_year'] = ("time", pd.to_datetime(snow_pillows_ds['time']).map(easysnowdata.utils.datetime_to_WY))
snow_pillows_ds.coords['DOWY'] = ("time", pd.to_datetime(snow_pillows_ds['time']).map(easysnowdata.utils.datetime_to_DOWY))
snow_pillows_ds

In [ ]:
snow_pillows_ds = snow_pillows_ds.sel(time=snow_pillows_ds['water_year'].isin(TARGET_WATER_YEARS))
snow_pillows_ds

## Station selection

The same physical station can appear twice in the archive: AWDB re-serves some CCSS and BC Snow Survey stations under its "MSNT" network label. Prefer the native clients (`cdec`, `databc`) — their archives are longer — and drop MSNT entries within 200 m of a native CCSS/BCSS station. Then drop stations with no SWE observations in the target water years (snow-depth-only sites etc.).

In [ ]:
snow_pillow_gdf = gpd.GeoDataFrame(
    {
        'station_id':   snow_pillows_ds['station_id'].values,
        'station_name': snow_pillows_ds['station_name'].values,
        'network':      snow_pillows_ds['network'].values,
        'client':       snow_pillows_ds['client'].values,
        'state':        snow_pillows_ds['state'].values,
        'elevation':    snow_pillows_ds['elevation'].values,
    },
    geometry=gpd.points_from_xy(snow_pillows_ds['longitude'].values, snow_pillows_ds['latitude'].values),
    crs='EPSG:4326',
)

msnt_gdf   = snow_pillow_gdf[snow_pillow_gdf['network'] == 'MSNT'].to_crs(epsg=3857)
native_gdf = snow_pillow_gdf[snow_pillow_gdf['network'].isin(['CCSS', 'BCSS'])].to_crs(epsg=3857)

joined = gpd.sjoin_nearest(msnt_gdf, native_gdf, distance_col='dist_m', lsuffix='left', rsuffix='right')
duplicate_msnt_ids = joined.loc[joined['dist_m'] < 200, 'station_id_left'].unique()
print(f"Dropping {len(duplicate_msnt_ids)} MSNT stations that duplicate native CCSS/BCSS stations")

snow_pillows_ds = snow_pillows_ds.sel(station_id=~snow_pillows_ds['station_id'].isin(duplicate_msnt_ids))
snow_pillow_gdf = snow_pillow_gdf[~snow_pillow_gdf['station_id'].isin(duplicate_msnt_ids)].reset_index(drop=True)

In [ ]:
swe_da = snow_pillows_ds['swe'].load()

has_swe = swe_da.notnull().any(dim='time')
print(f"{int(has_swe.sum())}/{has_swe.sizes['station_id']} stations have SWE data in WY {TARGET_WATER_YEARS[0]}-{TARGET_WATER_YEARS[-1]}")
swe_da = swe_da.sel(station_id=has_swe)
swe_da

## Quality control

Daily pillow SWE series contain spikes, dropouts, and partial seasons. All thresholds are in mm of SWE:

1. Remove negative values
2. **Spike filter** — mask points where either adjacent day-to-day change is ≥ 200 mm (also masks points next to missing data)
3. **Data density** — require ≥ 10 valid observations in every centered 20-day window
4. **Per station-year checks** — NaN out a station's whole water year if it has > 30 missing days in Nov–Mar, any gap > 10 days between valid observations, or doesn't start *and* end (essentially) snow-free (first/last valid value ≤ 100 mm)
5. **Seasonal-snowpack check** — keep only station-years with SWE ≥ 100 mm for ≥ 55 days within some 60-day window

In [ ]:
# --- QC thresholds (SWE in mm) ---
MAX_JUMP_MM         = 200   # max plausible day-to-day SWE change
MIN_VALID_IN_WINDOW = 10    # valid obs required in every centered 2*MIN_VALID_IN_WINDOW-day window
WINTER_MONTHS       = [11, 12, 1, 2, 3]
MAX_WINTER_MISSING  = 30    # max missing days in WINTER_MONTHS per water year
MAX_GAP_DAYS        = 10    # max gap between valid obs per water year
MAX_ENDPOINT_SWE_MM = 100   # water year record must start and end (essentially) snow-free
SEASONAL_SWE_MM     = 100   # "real winter" check: >= this much SWE ...
SEASONAL_DAYS       = 55    # ... for at least this many days in some 60-day window

In [ ]:
%%time
swe_qc_da = swe_da.where(swe_da >= 0)

# spike filter: a point survives only if both adjacent day-to-day changes are small
abs_diffs = np.abs(swe_qc_da.diff(dim='time'))
jump_mask = (abs_diffs.shift(time=1) < MAX_JUMP_MM) & (abs_diffs.shift(time=-1) < MAX_JUMP_MM)
swe_qc_da = swe_qc_da.where(jump_mask)

# data density: >= MIN_VALID_IN_WINDOW valid obs in every centered 2*MIN_VALID_IN_WINDOW-day window
valid_mask = swe_qc_da.notnull()
rolling_valid = valid_mask.rolling(time=MIN_VALID_IN_WINDOW * 2, center=True).sum()
swe_qc_da = swe_qc_da.where(rolling_valid >= MIN_VALID_IN_WINDOW)

In [ ]:
def check_missing_data(group):
    """NaN out a station's whole water year if it has too much missing winter data,
    long gaps between valid observations, or doesn't start and end snow-free."""
    winter_mask = group.time.dt.month.isin(WINTER_MONTHS)
    winter_missing = group.where(winter_mask, drop=True).isnull().sum(dim='time')
    too_much_missing = winter_missing > MAX_WINTER_MISSING

    def has_large_gap(values):
        valid = ~np.isnan(values)
        if not valid.any():
            return True
        gaps = np.diff(group.time.values[valid])
        if gaps.size == 0:
            return False
        return bool(np.any(gaps / np.timedelta64(1, 'D') > MAX_GAP_DAYS))

    large_gaps = xr.apply_ufunc(
        has_large_gap,
        group,
        input_core_dims=[['time']],
        vectorize=True,
        output_dtypes=[bool],
    )

    # proper seasonal evolution: record must start and end (essentially) snow-free
    valid = group.notnull()
    any_valid = valid.any(dim='time')
    first_idx = valid.argmax(dim='time')
    last_idx = group.sizes['time'] - 1 - valid.isel(time=slice(None, None, -1)).argmax(dim='time')
    first_valid = group.isel(time=first_idx)
    last_valid = group.isel(time=last_idx)
    improper_evolution = ((first_valid > MAX_ENDPOINT_SWE_MM) | (last_valid > MAX_ENDPOINT_SWE_MM)) | ~any_valid

    bad_station_years = too_much_missing | large_gaps | improper_evolution
    # isel(time=first_idx) leaks station-indexed time/water_year/DOWY coords onto the
    # mask; drop them so where() doesn't corrupt the group's 1-D water_year coord
    bad_station_years = bad_station_years.drop_vars(['time', 'water_year', 'DOWY'], errors='ignore')
    return group.where(~bad_station_years)


def check_seasonal_snow_swe(group):
    """Keep only station-years with a real seasonal snowpack:
    SWE >= SEASONAL_SWE_MM for >= SEASONAL_DAYS days within some 60-day window."""
    sufficient_swe = (group >= SEASONAL_SWE_MM).rolling(time=60, center=True, min_periods=SEASONAL_DAYS).sum()
    keep = (sufficient_swe >= SEASONAL_DAYS).any(dim='time')
    return group.where(keep)

In [ ]:
%%time
swe_qc_da = swe_qc_da.groupby('water_year').map(check_missing_data)
swe_qc_da = swe_qc_da.groupby('water_year').map(check_seasonal_snow_swe)

In [ ]:
valid_station_years = swe_qc_da.notnull().groupby('water_year').max()
print(f"***{int(valid_station_years.sum())} valid station-years "
      f"across {int(valid_station_years.any(dim='water_year').sum())} stations "
      f"(of {valid_station_years.sizes['station_id']} stations x {valid_station_years.sizes['water_year']} water years)***")

In [ ]:
# QC effect on an example station
example_station = '679_WA_SNTL'  # Paradise, WA
f, ax = plt.subplots(figsize=(15, 4))
swe_da.sel(station_id=example_station).plot(ax=ax, label='raw', color='lightgray', linewidth=2.5)
swe_qc_da.sel(station_id=example_station).plot(ax=ax, label='after QC', color='blue', linewidth=1)
ax.set_title(f'{example_station}: QC effect')
ax.legend()

## Per-water-year max SWE value and %-of-max SWE timing

In [ ]:
max_swe_timing_ds = swe_qc_da.groupby("water_year").max().to_dataset(name='max_SWE_value')
max_swe_timing_ds

In [ ]:
def find_pct_max_timing(da, pct, dim='time', skipna=True):
    """Find the time when SWE last crosses below a percentage of max SWE"""
    max_val = da.max(dim=dim, skipna=skipna)
    threshold = max_val * pct
    # Create boolean mask of values above threshold
    above_thresh = xr.where(da >= threshold, 1, np.nan)
    # Find the last True value
    return above_thresh.sel(time=slice(None, None, -1)).swap_dims({'time':'DOWY'}).idxmax(dim="DOWY", skipna=True).where(lambda x: x>0) 

In [ ]:
pct_list = [1.0, 0.99, 0.95, 0.9, 0.5]
for pct in pct_list:
    pct_str = str(int(pct * 100))
    max_swe_timing_ds[f'{pct_str}pct_of_max_SWE_timing'] = swe_qc_da.groupby("water_year").map(lambda x: find_pct_max_timing(x, pct)).where(lambda x: x>0)

max_swe_timing_ds

In [ ]:
# keep only stations with at least one usable station-year
usable = max_swe_timing_ds['max_SWE_value'].notnull().any(dim='water_year')
max_swe_timing_ds = max_swe_timing_ds.sel(station_id=usable)
print(f"{int(usable.sum())} stations with at least one usable station-year")

MAX_SWE_TIMING_OUTPUT_ZARR_FILEPATH.parent.mkdir(parents=True, exist_ok=True)
max_swe_timing_ds.attrs['dataset_version'] = VERSION  # so a stray copy is self-identifying
max_swe_timing_ds.to_zarr(MAX_SWE_TIMING_OUTPUT_ZARR_FILEPATH, mode='w')
print(f"Written -> {MAX_SWE_TIMING_OUTPUT_ZARR_FILEPATH}")

In [ ]:
# Persist the manuscript-cited QC counts (Sect. 2.3) -- durable home for the
# numbers printed by the two cells above.
from global_snowmelt_runoff_onset.results import save_result_table

qc_counts_df = pd.DataFrame([{
    'n_stations_with_swe': int(valid_station_years.sizes['station_id']),
    'n_water_years': int(valid_station_years.sizes['water_year']),
    'n_valid_station_years': int(valid_station_years.sum()),
    'n_stations_with_any_valid_year': int(valid_station_years.any(dim='water_year').sum()),
    'n_stations_with_usable_max_swe_timing': int(usable.sum()),
}])
save_result_table(qc_counts_df, 'qc_station_counts', version=VERSION)
qc_counts_df

In [ ]:
snow_pillow_gdf = snow_pillow_gdf[snow_pillow_gdf['station_id'].isin(max_swe_timing_ds['station_id'].values)].reset_index(drop=True)
snow_pillow_gdf

In [ ]:
snow_pillow_gdf.explore(
    column='network',
    tooltip=['station_id', 'station_name', 'network', 'state', 'elevation'],
    marker_kwds={'radius': 4},
)

## Build per-station chip Zarr with relative coordinates

For each snow pillow station, extract a 25×25 pixel spatial chip (1000 m buffer, 80 m pixels) from the global SAR runoff onset Zarr. Instead of absolute UTM x/y coordinates — which differ per station and can't be stacked — store offsets in **meters from the station center** (`x_rel`, `y_rel`). Every chip then has identical coordinate arrays and can be concatenated along a `station_id` dimension in a single Zarr.

The loop is **resumable**: it checks which stations are already in the Zarr at startup and skips them.

Output: `data/comparison_datasets/runoff_onset_snow_pillow_station_chips.zarr`

In [ ]:
# now bring in the runoff onset data......
# (the config is loaded in the setup cell at the top so the output paths carry its version)
print(f"runoff onset dataset: {VERSION} "
      f"({'icechunk repo' if config.output_store_is_icechunk else 'legacy consolidated Zarr v2'})")

In [ ]:
# Reads whichever store generation this config names -- an icechunk repo for >= v10,
# the legacy consolidated Zarr v2 mapper for <= v9 -- so v9 stays reproducible.
#
# chunks=None (no dask) is essential here: this notebook samples ~1,200 tiny 25x25
# windows, and the default chunks='auto' builds one dask graph entry per on-disk
# chunk -- ~17 million of them for the v10 grid (256x256 chunks over
# 11 x 204800 x 499998). That graph alone costs ~2.9 GB and ~100 s before a byte is
# read, and it made the threaded extraction below OOM. Zarr's own lazy indexing
# fetches only the intersecting chunks: measured 1.0 GB instead of 3.9 GB and ~10 s
# per chip instead of ~31 s.
runoff_onset_ds = config.open_runoff_onset_dataset(chunks=None)
runoff_onset_ds

In [ ]:
def get_station_gdf(station_points_gdf, station_id, buffer_radius=None):
    station_gdf = station_points_gdf[station_points_gdf.station_id == station_id]
    station_epsg = station_gdf.estimate_utm_crs().to_epsg()
    station_gdf = station_gdf.to_crs(epsg=station_epsg)
    if buffer_radius:
        station_gdf['geometry'] = station_gdf.geometry.buffer(buffer_radius)
    return station_gdf

In [ ]:
from tqdm.auto import tqdm

# --- chip geometry ---
BUFFER_RADIUS = 1000   # meters — defines the chip footprint
PIXEL_SIZE    = 80     # meters — native SAR resolution
N_HALF        = 12     # pixels each side; chip size = 2*N_HALF+1 = 25
TARGET_REL    = np.arange(-N_HALF, N_HALF + 1) * PIXEL_SIZE  # [-960, -880, ..., 0, ..., 960] m

BATCH_SIZE  = 30

# Max station_id string length — used to normalise dtype across batches so Zarr
# doesn't raise "Mismatched dtypes" when appending (each batch may have a different
# longest ID, giving a different <UN dtype).
MAX_SID_LEN = max(len(s) for s in snow_pillow_gdf['station_id'].values)

In [ ]:
def extract_chip_with_relative_coords(
    runoff_onset_ds, station_points_gdf, station_id,
    buffer_radius=BUFFER_RADIUS, pixel_size=PIXEL_SIZE, target_rel=TARGET_REL,
):
    """
    Extract a spatial chip from the global runoff onset Zarr around a snow pillow station.

    Absolute UTM x/y are replaced with offsets (meters from station center) so that
    every chip shares identical x_rel/y_rel coordinate arrays and can be concatenated
    along a station_id dimension.

    The station's absolute UTM center coordinates and EPSG code are stored as scalar
    coordinates (station_utm_x, station_utm_y, station_utm_epsg). The full per-pixel
    absolute UTM coordinate arrays are stored as data variables x_abs (dims: x_rel)
    and y_abs (dims: y_rel), preserving the true reprojected pixel centers.

    Steps
    -----
    1. Clip and reproject the global Zarr to local UTM at exactly PIXEL_SIZE metres.
    2. Fetch auxiliary layers (fcf, dem, worldcover, snow_class) while the chip still
       carries absolute coordinates and a valid CRS (needed for reproject_match).
    3. Capture x_abs/y_abs as data variables (dims x/y) before converting coords;
       rename({'x': 'x_rel', 'y': 'y_rel'}) propagates the dim names automatically.
    4. Store the station UTM centre and EPSG as scalar coordinates.
    5. Convert x/y → x_rel/y_rel by subtracting the station centre.
    6. Snap to TARGET_REL via reindex so every chip has an identical coordinate grid.
    """
    # Buffer station in local UTM
    station_utm_gdf = get_station_gdf(station_points_gdf, station_id, buffer_radius=buffer_radius)
    station_x = float(station_utm_gdf.geometry.centroid.x.iloc[0])
    station_y = float(station_utm_gdf.geometry.centroid.y.iloc[0])
    station_crs = station_utm_gdf.crs

    # Expand clip box by one pixel on each side to guarantee edge pixels survive reindex
    pad = pixel_size
    minx, miny, maxx, maxy = station_utm_gdf.total_bounds
    padded_bounds = (minx - pad, miny - pad, maxx + pad, maxy + pad)

    # Clip from global Zarr and reproject to local UTM at a fixed pixel size
    chip = (
        runoff_onset_ds[['runoff_onset', 'temporal_resolution']]
        .rio.clip_box(*padded_bounds, crs=station_crs)
        .rio.reproject(station_crs, resolution=pixel_size)
        .sel(water_year=TARGET_WATER_YEARS)
        .astype('float32')
    )

    # Fetch auxiliary layers while the chip still has absolute coords + CRS
    bounds_4326 = chip.rio.transform_bounds('EPSG:4326')
    chip['fcf'] = (
        easysnowdata.remote_sensing
        .get_forest_cover_fraction(bounds_4326, mask_nodata=True)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.bilinear)
    )
    chip['dem'] = (
        easysnowdata.topography
        .get_copernicus_dem(bounds_4326, resolution=30)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.bilinear)
    )
    chip['worldcover'] = (
        easysnowdata.remote_sensing
        .get_esa_worldcover(bounds_4326, mask_nodata=True)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.nearest)
    )
    chip['snow_class'] = (
        easysnowdata.remote_sensing
        .get_seasonal_snow_classification(bounds_4326, mask_nodata=True)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.nearest)
    )

    # Preserve original absolute UTM pixel coords as data variables *before* rename.
    # rename({'x': 'x_rel', 'y': 'y_rel'}) propagates dims automatically so that
    # x_abs ends up with dim 'x_rel' and y_abs with dim 'y_rel' — no recalculation.
    chip['x_abs'] = xr.DataArray(chip.x.values, dims=['x'])
    chip['y_abs'] = xr.DataArray(chip.y.values, dims=['y'])

    # Record the absolute UTM centre as scalar coords
    chip = chip.assign_coords(
        station_utm_x    = station_x,
        station_utm_y    = station_y,
        station_utm_epsg = station_crs.to_epsg(),
    )

    # Convert absolute x/y → relative coordinates (metres from station centre)
    chip = (
        chip
        .assign_coords(x=chip.x.values - station_x, y=chip.y.values - station_y)
        .rename({'x': 'x_rel', 'y': 'y_rel'})
    )

    # Snap to the fixed target grid; any remaining edge gap fills with NaN.
    # x_abs/y_abs are reindexed the same way, preserving the true pixel centers.
    chip = chip.reindex(
        x_rel=target_rel, y_rel=target_rel,
        method='nearest', tolerance=pixel_size // 2,
    )

    chip = chip.drop_vars('spatial_ref', errors='ignore')

    # Strip all variable and dataset attrs — rioxarray/reproject_match can leave
    # non-JSON-serializable objects (e.g. function references) that break Zarr writes.
    for var in list(chip.data_vars) + list(chip.coords):
        chip[var].attrs = {}
    chip.attrs = {}

    return chip

In [ ]:
#!rm -rf {RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH}

In [ ]:
# --- Resume check ---
# Run this cell at the start of each session to determine which stations remain.
RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH.parent.mkdir(parents=True, exist_ok=True)

if RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH.exists():
    try:
        _existing = xr.open_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
        already_done = set(_existing.station_id.values)
        _existing.close()
        zarr_initialized = True
        print(f"Resuming: {len(already_done)}/{len(snow_pillow_gdf)} ({100*len(already_done)/len(snow_pillow_gdf):.2f}%) stations already written")
        print(140*'-')
        print("Existing Zarr contents:")
        print(_existing)
    except Exception as e:
        print(f"Existing Zarr is corrupt ({e}); deleting and starting fresh.")
        shutil.rmtree(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
        already_done = set()
        zarr_initialized = False
else:
    already_done = set()
    zarr_initialized = False
    print("Starting fresh")

remaining_ids = [sid for sid in snow_pillow_gdf['station_id'].values if sid not in already_done]
print(140*'-')
print(f"{len(remaining_ids)} stations remaining")

In [ ]:
import warnings
import rasterio.errors
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc

N_WORKERS = 12   # I/O-bound; lower to 4 if rate-limit errors appear from Zenodo/Planetary Computer

chunk_spec = {
    'station_id': BATCH_SIZE,
    'water_year': len(TARGET_WATER_YEARS),
    'y_rel': len(TARGET_REL),
    'x_rel': len(TARGET_REL),
}

def _extract_one(sid):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
            chip = extract_chip_with_relative_coords(runoff_onset_ds, snow_pillow_gdf, sid)

        chip = chip.expand_dims('station_id').assign_coords(station_id=[sid])

        # Bind scalar station coords to station_id dim so xr.concat always produces
        # ('station_id',) dims. Without this, xr.concat (coords='different') collapses
        # them to scalar () when all chips in a batch share the same UTM zone, which
        # mismatches the ('station_id',) dims already on disk → ValueError on append.
        chip = chip.assign_coords(
            station_utm_x    = ('station_id', [float(chip.station_utm_x)]),
            station_utm_y    = ('station_id', [float(chip.station_utm_y)]),
            station_utm_epsg = ('station_id', [int(chip.station_utm_epsg)]),
        )

        # Materialise into numpy — clears dask graph references so Azure read memory
        # is released immediately rather than accumulating across concurrent workers.
        chip = chip.load()
        return sid, chip, None
    except Exception as e:
        return sid, None, e

for batch_idx, batch_start in enumerate(
    tqdm(range(0, len(remaining_ids), BATCH_SIZE), desc="Batches")
):
    batch_ids = remaining_ids[batch_start : batch_start + BATCH_SIZE]
    batch_chips = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(_extract_one, sid): sid for sid in batch_ids}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"  Batch {batch_idx}", leave=False):
            sid, chip, err = future.result()
            if err:
                print(f"  Skipped {sid}: {err}")
            else:
                batch_chips.append(chip)

    if not batch_chips:
        continue

    # coords='all' forces station_utm_x/y/epsg to be concatenated along station_id
    # even when all chips in a batch share the same UTM zone value. Without this,
    # xr.concat collapses identical coords to scalar (), mismatching dims on disk.
    batch_ds = xr.concat(batch_chips, dim='station_id', coords='all').chunk(chunk_spec)

    # Normalise station_id to a fixed-width string dtype so every batch matches
    # the first write. xarray infers <UN from the longest ID in each batch, which
    # varies, causing Zarr to raise "Mismatched dtypes" on append.
    batch_ds = batch_ds.assign_coords(
        station_id=batch_ds.station_id.values.astype(f'U{MAX_SID_LEN}')
    )

    if not zarr_initialized:
        batch_ds.attrs['dataset_version'] = VERSION  # so a stray copy is self-identifying
        batch_ds.to_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH, mode='w')
        zarr_initialized = True
    else:
        batch_ds.to_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH, append_dim='station_id')

    del batch_chips, batch_ds
    gc.collect()

print(f"Done — {RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH}")

In [ ]:
# Verification
chips_ds = xr.open_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
print(chips_ds)

assert np.array_equal(chips_ds.x_rel.values, TARGET_REL), "x_rel mismatch"
assert np.array_equal(chips_ds.y_rel.values, TARGET_REL), "y_rel mismatch"
print(f"\nx_rel/y_rel identical across all stations ✓  ({len(TARGET_REL)} values: {TARGET_REL[0]}…{TARGET_REL[-1]} m)")

chips_ds['runoff_onset'].sel(station_id='679_WA_SNTL').plot.imshow(col='water_year', col_wrap=5)

## Combine chip Zarr and SWE timing Zarr

After both Zarrs are complete, merge them in memory (or write a combined Zarr).
`max_swe_timing_ds` has dims `(station_id, water_year)` and shares the `station_id` dimension with the chip dataset, so `xr.merge` aligns on both.

In [ ]:
runoff_onset_chips_ds = xr.open_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
runoff_onset_chips_ds

In [ ]:
max_swe_timing_ds = xr.open_zarr(MAX_SWE_TIMING_OUTPUT_ZARR_FILEPATH)
max_swe_timing_ds

In [ ]:
snow_pillow_chips_and_max_swe_timing_ds = xr.merge([runoff_onset_chips_ds, max_swe_timing_ds], join='inner')
snow_pillow_chips_and_max_swe_timing_ds